# 🛰️ ISRO Bharatiya Antariksh Hackathon
## Dual-Stage Thermal Infrared (TIRS) to Ultra-HD True Color RGB Synthesis
Welcome to the official training and inference notebook. Optimized for **Google Colab GPUs (T4 / V100 / A100)** with **Automatic Mixed Precision (AMP FP16)** and **Direct Google Drive Checkpoint Sync**.

### Recommended Setup:
1. In the top menu, select **Runtime** -> **Change runtime type**.
2. Under Hardware Accelerator, select **T4 GPU** (or any GPU) and click Save.
3. Run the cells step-by-step below.

In [ ]:
# 1. Hardware Verification
import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Running on CPU or TPU. For fastest training, select Runtime -> Change runtime type -> T4 GPU.")

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clone / Setup Repository in Colab
import os
%cd /content
# Clone or pull latest repository updates
!git clone https://github.com/vinay7225/IR-RGB.git
%cd /content/IR-RGB

In [ ]:
# 4. Install Dependencies
!pip install -q -r requirements.txt
print("✅ Dependencies successfully installed!")

In [ ]:
# 5. Fast Dataset Setup
# Copying a single zip file is 100x faster than copying thousands of individual .npz files!
import os, glob

!mkdir -p FILES

zip_candidates = glob.glob('/content/drive/MyDrive/**/processed.zip', recursive=True)
if zip_candidates:
    print(f"Found compressed dataset at {zip_candidates[0]}. Unzipping directly to local disk...")
    !unzip -q {zip_candidates[0]} -d FILES/
else:
    drive_processed = glob.glob('/content/drive/MyDrive/**/processed', recursive=True)
    if drive_processed:
        print(f"Copying {drive_processed[0]} to local disk...")
        !cp -r {drive_processed[0]} FILES/
    else:
        print("Notice: Please place your 'processed' folder or 'processed.zip' in Google Drive.")

!ls -lh FILES/processed

In [ ]:
# 6. Configure Paths & Direct Google Drive Checkpoint Auto-Save
# This automatically detects where your uploaded checkpoints live on Google Drive!
import yaml, os, glob

# 1. Search for your uploaded checkpoints on Google Drive
found_chks = glob.glob('/content/drive/MyDrive/**/pix2pix_gen_*.pth', recursive=True) or glob.glob('/content/drive/MyDrive/**/realesrgan_*.pth', recursive=True)

if found_chks:
    GDRIVE_CHKS = os.path.dirname(found_chks[0])
    print(f"🎯 Found your uploaded checkpoints on Google Drive in: {GDRIVE_CHKS}")
    print(f"   Detected files: {[os.path.basename(f) for f in glob.glob(os.path.join(GDRIVE_CHKS, '*.pth'))][:5]}...")
else:
    GDRIVE_CHKS = '/content/drive/MyDrive/ISRO_Hackathon_Checkpoints'
    os.makedirs(GDRIVE_CHKS, exist_ok=True)
    print(f"Checkpoints directory initialized at: {GDRIVE_CHKS}")

with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['data']['raw_dir'] = 'FILES'
config['data']['processed_dir'] = 'FILES/processed'
# Direct save path to Google Drive:
config['training']['checkpoints_dir'] = GDRIVE_CHKS
config['training']['outputs_dir'] = os.path.join(GDRIVE_CHKS, 'outputs')
os.makedirs(config['training']['outputs_dir'], exist_ok=True)

with open('configs/config.yaml', 'w') as f:
    yaml.dump(config, f)

print(f"✅ Configured: Any newly trained epoch will save DIRECTLY into your Google Drive: {GDRIVE_CHKS}")

--- 
## 🏋️ Stage 1: Train Real-ESRGAN (Thermal Super-Resolution)
If your uploaded checkpoint is already at Epoch 10, the script will automatically recognize it is finished and ready!

In [ ]:
# Run Real-ESRGAN Training (with AMP FP16 & Direct Drive Sync)
os.environ['PYTHONPATH'] = '.'
!python training/train_realesrgan.py

--- 
## 🎨 Stage 2: Train Pix2Pix GAN (Physical Colorization)
Automatically detects your uploaded `pix2pix_gen_epoch_26.pth` on Google Drive and starts training directly from **Epoch 27** onwards!

In [ ]:
# Run Pix2Pix Colorization Training
os.environ['PYTHONPATH'] = '.'
!python training/train_pix2pix.py

--- 
## 👁️ Inspect Visual Validation Samples Generated During Training

In [ ]:
import glob, os, yaml
from IPython.display import Image, display

with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)
outputs_dir = config['training'].get('outputs_dir', 'outputs')
samples = sorted(glob.glob(os.path.join(outputs_dir, 'samples/*.png')))

if samples:
    print(f"Latest Validation Sample: {samples[-1]} [Thermal | Generated RGB | Ground Truth]")
    display(Image(samples[-1]))
else:
    print("No sample images generated yet. Run training to generate samples.")

--- 
## 🔍 Stage 3: Full-Scene Ultra-HD Inference with Deep Zoom Viewer

In [ ]:
# Run seamless Hann-feathered Ultra HD inference on your test scene
import glob
from inference.predict import run_inference

raw_scenes = sorted(glob.glob('FILES/LC08*'))
if raw_scenes:
    test_scene = raw_scenes[0]
    print(f"Synthesizing Ultra HD scene: {test_scene}")
    res = run_inference(test_scene)
    print(f"Ultra HD GeoTIFF: {res['output_tif']}")
    print(f"Deep Zoom Viewer: {res['viewer_html']}")
else:
    print("No raw scene found in FILES. You can run inference on your test scenes.")

In [ ]:
# Launch Interactive Deep Zoom Viewer directly inside Google Colab
from IPython.display import IFrame, display
import glob

viewers = glob.glob('outputs/*_viewer.html') or glob.glob('/content/drive/MyDrive/**/*_viewer.html', recursive=True)
if viewers:
    display(IFrame(src=viewers[-1], width='100%', height=600))
else:
    print("Run inference first to generate the Ultra HD deep zoom viewer.")